In [ ]:
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T

In [ ]:
session.use_database("AIRBNB_INVESTMENT_DB")
session.use_schema("BRONZE")

In [ ]:
table_list = (
    session.sql("""
        SELECT TABLE_NAME 
        FROM AIRBNB_INVESTMENT_DB.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = 'BRONZE'
        AND TABLE_NAME LIKE '%_CALENDAR'
    """).collect()
)

cities = [row["TABLE_NAME"].replace("_CALENDAR", "").lower() for row in table_list]
print(f"Found {len(cities)} cities: {cities}")

In [ ]:
for city in cities:
    df = session.table(f"bronze.{city}_calendar")

In [ ]:
expected_cols = {"listing_id", "date", "is_booked", "minimum_nights", "maximum_nights"}
actual_cols   = set(df.columns)
missing_cols  = expected_cols - actual_cols

if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")
else:
    print("Schema check passed — all expected columns present")

# Cast columns to expected types
df = (
    df
    .withColumn("listing_id",     F.col("listing_id").cast(T.LongType()))
    .withColumn("date",           F.to_date(F.col("date")))
    .withColumn("minimum_nights", F.col("minimum_nights").cast(T.IntegerType()))
    .withColumn("maximum_nights", F.col("maximum_nights").cast(T.IntegerType()))
)

df = df.withColumn(
    "is_booked",
    F.when(F.lower(F.col("is_booked").cast(T.StringType())).isin("t", "true",  "1", "yes"), F.lit(True))
     .when(F.lower(F.col("is_booked").cast(T.StringType())).isin("f", "false", "0", "no"),  F.lit(False))
     .otherwise(None)
     .cast(T.BooleanType())
)

In [ ]:
critical_cols     = ["listing_id", "date", "is_booked"]
non_critical_cols = ["minimum_nights", "maximum_nights"]

print("\nNull counts per column")
for col in critical_cols + non_critical_cols:
    null_count = df.filter(F.col(col).isNull()).count()
    print(f"{col}: {null_count} nulls")

before_null_drop = df.count()
df = df.filter(
    F.col("listing_id").isNotNull() &
    F.col("date").isNotNull()       &
    F.col("is_booked").isNotNull()
)
after_null_drop = df.count()
print(f"\nDropped {before_null_drop - after_null_drop} rows with nulls in critical columns")


In [ ]:
total_rows    = df.count()
distinct_rows = df.dropDuplicates(["listing_id", "date"]).count()
dupe_count    = total_rows - distinct_rows

if dupe_count > 0:
    print(f"Found {dupe_count} duplicate (listing_id, date) rows — keeping first occurrence")
    from snowflake.snowpark.window import Window
    dedup_window = Window.partitionBy("listing_id", "date").orderBy("listing_id")
    df = (
        df
        .withColumn("_row_num", F.row_number().over(dedup_window))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
    )
else:
    print("No duplicate (listing_id, date) combinations found")

In [ ]:
future_date   = F.lit("2026-12-31").cast(T.DateType())
airbnb_launch = F.lit("2008-01-01").cast(T.DateType())

early_dates  = df.filter(F.col("date") < airbnb_launch).count()
future_dates = df.filter(F.col("date") > future_date).count()

if early_dates > 0:
    print(f"{early_dates} calendar rows dated before Airbnb launched (pre 2008)")
if future_dates > 0:
    print(f"{future_dates} calendar rows with dates beyond 2026")

df = df.filter(
        (F.col("date") >= airbnb_launch) &
        (F.col("date") <= future_date)
)

In [ ]:
invalid_min = df.filter(F.col("minimum_nights") <= 0).count()
invalid_max = df.filter(F.col("maximum_nights") <= 0).count()

if invalid_min > 0:
    print(f"{invalid_min} rows with minimum_nights <= 0")
if invalid_max > 0:
    print(f"{invalid_max} rows with maximum_nights <= 0")

df = (
        df
        .filter(
            F.col("minimum_nights").isNull() | (F.col("minimum_nights") > 0)
        )
        .filter(
            F.col("maximum_nights").isNull() | (F.col("maximum_nights") > 0)
        )
    )

invalid_range = df.filter(
    F.col("minimum_nights").isNotNull() &
    F.col("maximum_nights").isNotNull() &
    (F.col("minimum_nights") > F.col("maximum_nights"))
).count()

if invalid_range > 0:
    print(f"{invalid_range} rows where minimum_nights > maximum_nights")

df = df.filter(
        F.col("minimum_nights").isNull() |
        F.col("maximum_nights").isNull() |
        (F.col("minimum_nights") <= F.col("maximum_nights"))
    )

high_min_nights = df.filter(F.col("minimum_nights") > 365).count()
if high_min_nights > 0:
    print(f"{high_min_nights} rows with minimum_nights > 365")

df = df.filter(
        F.col("minimum_nights").isNull() |
        (F.col("minimum_nights") <= 365)
    )

In [ ]:
negative_listings = df.filter(F.col("listing_id") <= 0).count()

if negative_listings > 0:
    print(f"{negative_listings} rows with invalid listing_id <= 0 — dropping")

df = df.filter(F.col("listing_id") > 0)

In [ ]:
silver = df.groupBy("listing_id").agg(
        F.count("date").alias("total_calendar_days"),
        F.sum(F.when(F.col("is_booked"), 1).otherwise(0)).alias("days_booked"),
        F.min("date").alias("calendar_start_date"),
        F.max("date").alias("calendar_end_date"),
        F.avg("minimum_nights").alias("avg_minimum_nights"),
        F.avg("maximum_nights").alias("avg_maximum_nights"),
    )

In [ ]:
silver = silver.withColumn(
        "occupancy_rate",
        F.when(
            F.col("total_calendar_days") > 0,
            F.round(F.col("days_booked") / F.col("total_calendar_days"), 4)
        ).otherwise(None)
    )

    silver = silver.withColumn(
        "date_range",
        F.concat(
            F.date_format(F.col("calendar_start_date"), "dd/MM/yyyy"),
            F.lit(" - "),
            F.date_format(F.col("calendar_end_date"),   "dd/MM/yyyy"),
        )
    )

    silver = silver.withColumn(
        "occupancy_quality_flag",
        F.when(F.col("occupancy_rate") == 1.0, F.lit("FULLY_BOOKED_CHECK"))
         .when(F.col("occupancy_rate") == 0.0, F.lit("ZERO_BOOKINGS"))
         .otherwise(F.lit("OK"))
    )

    silver = silver.select(
        "listing_id",
        "total_calendar_days",
        "days_booked",
        "occupancy_rate",
        "calendar_start_date",
        "calendar_end_date",
        "date_range",
        "avg_minimum_nights",
        "avg_maximum_nights",
        "occupancy_quality_flag",
    )


In [ ]:
session.use_schema("SILVER")
output_table = f"silver_{city}_calendar"
silver.write.mode("overwrite").save_as_table(output_table)
print(f"Saved to AIRBNB_INVESTMENT_APP.SILVER.{output_table}")